# Step 2: Data Cleaning

In Step 1 we found four issues: exact duplicate rows, missing `Description` values, cancellations (negative quantities), and rows with a price of zero or less. Here we decide what to do about each one — and explain *why*, not just *what*.

**Rule of thumb for this project:** only remove a row if we have a concrete reason to distrust it or if it can't answer a business question (e.g. we can't attribute revenue to "unknown product"). Every decision below is documented so it can be questioned and reproduced.

In [1]:
import pandas as pd

df = pd.read_excel("data/online_retail.xlsx")
print("Starting rows:", len(df))

Starting rows: 541909


## 1. Remove exact duplicate rows

A *duplicate row* here means every single column (invoice, product, quantity, date, price, customer) is identical. That can't be two different real purchases — a customer buying the same product twice in one order would normally show up as one line with `Quantity=2`, not two identical lines. This pattern looks like a scanning or logging error in the checkout system, so we remove the repeat and keep one copy of each.

In [2]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows -> {len(df)} rows remain")

Removed 5268 duplicate rows -> 536641 rows remain


## 2. Drop rows with missing product description

We checked: every row with a missing `Description` also has `UnitPrice = 0`. These look like internal stock adjustments (e.g. damaged goods write-offs), not real customer sales — and without a product name we can't assign them to any category anyway. Since it's a small slice (~0.3% of rows), we drop them rather than guess what they were.

In [3]:
before = len(df)
df = df.dropna(subset=["Description"])
print(f"Removed {before - len(df)} rows with missing Description -> {len(df)} rows remain")

Removed 1454 rows with missing Description -> 535187 rows remain


## 3. Separate returns/cancellations from completed sales

A negative `Quantity` means a return or cancellation, not a sale. For a **revenue** analysis ("how much did we sell"), mixing in negative quantities would understate real sales performance and answer a different question ("net sales after returns"). So we split the data in two:

- `sales`: completed sales (`Quantity > 0`) — used for all revenue analysis in this project
- `returns`: returns/cancellations, kept aside as a separate metric (return rate), not deleted

We're not throwing the returns away — just answering "how much revenue did we generate" and "how many returns did we get" as two separate, honest questions instead of blending them into one number.

In [4]:
returns = df[df["Quantity"] <= 0].copy()
sales = df[df["Quantity"] > 0].copy()

return_rate = len(returns) / len(df)
print(f"Returns/cancellations: {len(returns)} rows ({return_rate*100:.1f}% of all transactions)")
print(f"Completed sales: {len(sales)} rows")

Returns/cancellations: 9725 rows (1.8% of all transactions)
Completed sales: 525462 rows


## 4. Remove sales rows with a price of zero or less

Within the completed-sales set, a handful of rows still have `UnitPrice <= 0`. A real sale always has a positive price, so these are most likely free promotional samples or data entry errors — either way, they'd distort revenue totals (a $0 "sale" of thousands of units, for example), so we drop them.

In [5]:
before = len(sales)
sales = sales[sales["UnitPrice"] > 0].copy()
print(f"Removed {before - len(sales)} zero/negative-price rows -> {len(sales)} rows remain")

Removed 584 zero/negative-price rows -> 524878 rows remain


## 5. What we're keeping as-is: missing `CustomerID`

~25% of sales rows have no `CustomerID` (likely guest checkouts). We deliberately do **not** drop these — the transaction and its revenue are still real, we just can't tie it to a specific customer. Any *customer-level* analysis later will need to filter these out explicitly, but the revenue/product/country analysis in this project doesn't need `CustomerID` at all.

In [6]:
missing_customer = sales["CustomerID"].isna().mean()
print(f"Missing CustomerID in final sales set: {missing_customer*100:.1f}%")

Missing CustomerID in final sales set: 25.2%


## 6. Add a `Revenue` column

This dataset has no ready-made revenue figure — just quantity and unit price. `Revenue = Quantity x UnitPrice` is the number every later chart in this project is built on.

In [7]:
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]

print("Final clean dataset:", sales.shape)
print(f"Total revenue: £{sales['Revenue'].sum():,.2f}")
sales.head()

Final clean dataset: (524878, 9)
Total revenue: £10,642,110.80


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Save the cleaned dataset

We save this as a CSV so the next notebooks (analysis, charts) can just load it directly instead of repeating all the cleaning steps above.

In [8]:
sales.to_csv("data/sales_clean.csv", index=False)
returns.to_csv("data/returns.csv", index=False)
print("Saved data/sales_clean.csv and data/returns.csv")

Saved data/sales_clean.csv and data/returns.csv


## Summary

| Step | Rows removed | Reason |
|---|---|---|
| Exact duplicates | 5,268 | Logging/scanning errors |
| Missing description | 1,454 | Can't identify product, price was always £0 anyway |
| Returns/cancellations | 9,725 | Set aside as a separate metric, not deleted |
| Zero/negative price | 584 | Not real sales revenue |
| **Final clean sales dataset** | **524,878 rows kept** | out of 541,909 original rows (96.9%) |

Every number in this table comes directly from the code above — nothing here was estimated or invented.